# JMIR evidence-selection experiment — automated integrity validation and freeze

This notebook performs objective quality-control checks and freezes the 125-case evaluation set. It does **not** claim independent clinical reannotation. Reference answers are retained from VQA-RAD and SLAKE.

Validation scope: source-row linkage, image-file integrity, unique identifiers, dataset and answer allocation, binary-answer eligibility, and multilingual-field completeness. A GPU and Ollama are not required.


## 0. Mount Drive and load the corrected casebook

Use a normal CPU runtime. Do not install or upgrade pandas, PyArrow, or `datasets` in this notebook.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import hashlib, json
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK_DIR = ROOT / 'casebooks'
CASEBOOK_DIR.mkdir(parents=True, exist_ok=True)
SOURCE = CASEBOOK_DIR / 'candidate_casebook_evaluation_linguistically_corrected.csv'
if not SOURCE.exists():
    matches = list(ROOT.rglob('candidate_casebook_evaluation_linguistically_corrected.csv'))
    if not matches:
        raise FileNotFoundError('The linguistically corrected casebook was not found under ROOT.')
    SOURCE = max(matches, key=lambda p: p.stat().st_mtime)

df = pd.read_csv(SOURCE, dtype=str, keep_default_na=False)
df.columns = [str(c).strip() for c in df.columns]
for col in ['phase','dataset','reference_answer','image_saved']:
    df[col] = df[col].astype(str).str.strip().str.lower()
evaluation = df[df['phase'].eq('evaluation')].copy().sort_values('case_id').reset_index(drop=True)
SOURCE_SHA256 = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
print('Source:', SOURCE)
print('SHA256:', SOURCE_SHA256)
print('Rows:', len(df), '| Evaluation:', len(evaluation))


Mounted at /content/drive
Source: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook_evaluation_linguistically_corrected.csv
SHA256: 8121a111f30cde517cbece5e5aeaa90f6eb3f36938c9ad33b5120c921036ed77
Rows: 135 | Evaluation: 125


## 1. Structural, allocation, and linguistic-integrity checks

These checks preserve the experimental design without altering any labels.


In [3]:
required = [
    'case_id','dataset','split','source_index','image_key','image_name','question_en',
    'reference_answer','phase','image_path','image_saved','proposition_en',
    'question_id','question_ms','proposition_id','proposition_ms'
]
missing_columns = [c for c in required if c not in evaluation.columns]
assert not missing_columns, f'Missing columns: {missing_columns}'
assert len(evaluation) == 125, f'Expected 125 evaluation cases, found {len(evaluation)}.'
dataset_counts = evaluation['dataset'].value_counts().to_dict()
answer_counts = evaluation['reference_answer'].value_counts().to_dict()
assert dataset_counts == {'vqa_rad': 100, 'slake': 25}, dataset_counts
assert set(answer_counts) == {'yes','no'} and sum(answer_counts.values()) == 125, answer_counts
assert abs(answer_counts['yes'] - answer_counts['no']) <= 1, answer_counts
assert evaluation['case_id'].is_unique, 'Duplicate case_id.'
assert evaluation['image_key'].is_unique, 'Duplicate image_key.'
assert evaluation['image_path'].is_unique, 'Duplicate image_path.'

text_columns = ['question_en','reference_answer','proposition_en','question_id','question_ms','proposition_id','proposition_ms','image_path']
missing_text = {c: int(evaluation[c].astype(str).str.strip().eq('').sum()) for c in text_columns}
assert not any(missing_text.values()), f'Missing required values: {missing_text}'
joined_language = evaluation[['question_id','question_ms','proposition_id','proposition_ms']].astype(str).agg(' | '.join, axis=1).str.lower()
for token in ['¿','pneumotoksis','sisyen','bengkung','apropiasi','estudio de cuello','hay evidencia']:
    hits = evaluation.loc[joined_language.str.contains(token, regex=False), 'case_id'].tolist()
    assert not hits, f'Residual linguistic corruption {token!r}: {hits}'
print('Structural and linguistic-integrity checks passed.')
print('Dataset counts:', dataset_counts)
print('Answer counts:', answer_counts)


Structural and linguistic-integrity checks passed.
Dataset counts: {'vqa_rad': 100, 'slake': 25}
Answer counts: {'no': 63, 'yes': 62}


## 2. Verify every selected row against the original benchmark record

This confirms that no question, answer, image name, source index, or recorded image key drifted during casebook preparation.


In [4]:
try:
    from datasets import load_dataset
except Exception as e:
    raise RuntimeError(
        'The preinstalled datasets/PyArrow environment is inconsistent. Restart the Colab session, '
        'then rerun this notebook without running any pip installation cell.'
    ) from e

print('Loading source benchmarks...')
vqa = load_dataset('flaviagiammarino/vqa-rad')
slake = load_dataset('bokelvin/slake')
sources = {'vqa_rad': vqa, 'slake': slake}

def image_fingerprint(image, fallback):
    if isinstance(image, Image.Image):
        normalized = image.convert('RGB')
        payload = (str(normalized.size) + '|RGB|').encode() + normalized.tobytes()
        return hashlib.sha1(payload).hexdigest()[:16]
    if isinstance(image, dict) and image.get('bytes'):
        return hashlib.sha1(image['bytes']).hexdigest()[:16]
    return hashlib.sha1(str(fallback).encode('utf-8')).hexdigest()[:16]

audit = []
for _, row in evaluation.iterrows():
    rec = {'case_id': row['case_id'], 'dataset': row['dataset']}
    try:
        split = row['split']
        source_index = int(row['source_index'])
        raw = sources[row['dataset']][split][source_index]
        raw_question = str(raw.get('question') or raw.get('Question') or '').strip()
        raw_answer = str(raw.get('answer') or raw.get('Answer') or '').strip().lower().rstrip('.')
        raw_name = str(raw.get('img_name') or raw.get('image_name') or raw.get('image_id') or raw.get('img_id') or f'{split}_{source_index}')
        raw_key = image_fingerprint(raw.get('image'), raw_name)
        rec.update({
            'question_match': raw_question == row['question_en'].strip(),
            'answer_match': raw_answer == row['reference_answer'],
            'image_name_match': raw_name == row['image_name'],
            'image_key_match': raw_key == row['image_key'],
            'source_link_ok': raw_question == row['question_en'].strip() and raw_answer == row['reference_answer'] and raw_name == row['image_name'] and raw_key == row['image_key'],
            'error': ''
        })
    except Exception as e:
        rec.update({'question_match':False,'answer_match':False,'image_name_match':False,'image_key_match':False,'source_link_ok':False,'error':repr(e)})
    audit.append(rec)

audit = pd.DataFrame(audit)
bad_source_links = audit[~audit['source_link_ok']]
if len(bad_source_links):
    display(bad_source_links)
    raise RuntimeError(f'{len(bad_source_links)} cases failed source-record verification.')
print('Source-record linkage verified:', int(audit['source_link_ok'].sum()), '/ 125')


Loading source benchmarks...


README.md:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

data/train-00000-of-00001-eb8844602202be(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-eb8844602202be(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-e5bc3d208bb4dee(…): reconstructing file:   0%|          |  0.00B / 10.3MB            

...

test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Source-record linkage verified: 125 / 125


## 3. Verify all saved image files

This verifies existence and decodability. It does not perform medical interpretation.


In [5]:
image_results = []
for _, row in evaluation.iterrows():
    path = ROOT / row['image_path']
    rec = {'case_id':row['case_id'],'image_path':row['image_path'],'exists':path.is_file(),'decodable':False,'width':'','height':'','error':''}
    if path.is_file():
        try:
            with Image.open(path) as im:
                rec['width'], rec['height'] = im.size
                im.verify()
            rec['decodable'] = True
        except Exception as e:
            rec['error'] = repr(e)
    image_results.append(rec)
image_results = pd.DataFrame(image_results)
bad_images = image_results[~image_results['exists'] | ~image_results['decodable']]
if len(bad_images):
    display(bad_images)
    raise RuntimeError(f'{len(bad_images)} images failed integrity validation.')
assert evaluation['image_saved'].eq('true').all()
print('Saved images verified:', int(image_results['decodable'].sum()), '/ 125')


Saved images verified: 125 / 125


## 4. Freeze the evaluation casebook

This freeze records validation for experimental use, not independent clinical approval. Existing frozen content is protected from silent replacement.


In [6]:
frozen = evaluation.copy()
frozen['include_after_review'] = 'yes'
frozen['reviewer_initials'] = ''
frozen['review_status'] = 'approved_for_experiment'
frozen['clinical_validity_notes'] = (
    'Original benchmark reference answer retained; source linkage, file integrity, '
    'binary eligibility, and linguistic consistency validated. No independent clinical reannotation performed.'
)
assert len(frozen) == 125
assert frozen['dataset'].value_counts().to_dict() == {'vqa_rad':100,'slake':25}
assert frozen['reference_answer'].value_counts().to_dict() == {'no':63,'yes':62}
assert frozen['case_id'].is_unique and frozen['image_key'].is_unique and frozen['image_path'].is_unique

FROZEN = CASEBOOK_DIR / 'evaluation_cases_frozen.csv'
CHECKSUM = CASEBOOK_DIR / 'evaluation_cases_frozen.csv.sha256'
AUDIT = CASEBOOK_DIR / 'evaluation_integrity_audit.csv'
MANIFEST = CASEBOOK_DIR / 'evaluation_cases_frozen_manifest.json'
csv_bytes = frozen.to_csv(index=False, encoding='utf-8').encode('utf-8')
frozen_sha = hashlib.sha256(csv_bytes).hexdigest()
if FROZEN.exists() and FROZEN.read_bytes() != csv_bytes:
    raise FileExistsError(f'{FROZEN} already exists with different content. Rename/archive the earlier freeze explicitly before creating a new one.')
FROZEN.write_bytes(csv_bytes)
CHECKSUM.write_text(f'{frozen_sha}  {FROZEN.name}\n', encoding='utf-8')

full_audit = audit.merge(image_results, on='case_id', how='left', validate='one_to_one')
full_audit['validated_at_utc'] = datetime.now(timezone.utc).isoformat()
full_audit['source_casebook_sha256'] = SOURCE_SHA256
full_audit.to_csv(AUDIT, index=False, encoding='utf-8')
manifest = {
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'source_file':str(SOURCE),'source_sha256':SOURCE_SHA256,
    'frozen_file':str(FROZEN),'frozen_sha256':frozen_sha,'n_cases':125,
    'dataset_counts':frozen['dataset'].value_counts().to_dict(),
    'answer_counts':frozen['reference_answer'].value_counts().to_dict(),
    'validation_scope':['source-record linkage','saved-image existence and decodability','unique case/image identifiers','binary-answer eligibility','dataset and answer allocation','multilingual-field completeness'],
    'clinical_reannotation':False,
    'reference_answer_policy':'Retained from the source VQA-RAD and SLAKE benchmarks.'
}
MANIFEST.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Evaluation casebook frozen successfully.')
print('Frozen CSV:', FROZEN)
print('SHA256:', frozen_sha)
print('Integrity audit:', AUDIT)
print('Manifest:', MANIFEST)


Evaluation casebook frozen successfully.
Frozen CSV: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/evaluation_cases_frozen.csv
SHA256: c1617014f21be02ae1ec78ebb3cf6e862429a88552c0bc25c721b1fa07a9ab46
Integrity audit: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/evaluation_integrity_audit.csv
Manifest: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/evaluation_cases_frozen_manifest.json


## Methodological record

> Reference answers were retained from the source VQA-RAD and SLAKE benchmarks. We verified source-record linkage, file integrity, image uniqueness, binary-answer eligibility, and cross-language semantic consistency but did not independently clinically reannotate the images.

After the freeze succeeds, proceed to the main checkpointed 2250-trial inference notebook.
